## Data Cleaning

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Membaca sheet sales
sales = pd.read_excel(
    "FIX_RAW.xlsx",
    sheet_name="sales"
)

# Membaca daftar produk
with open("fix_data_produk.txt", "r", encoding="utf-8") as f:
    daftar_produk = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Jumlah data sales:", len(sales))
print("Jumlah produk dalam TXT:", len(daftar_produk))

Jumlah data sales: 3211
Jumlah produk dalam TXT: 45


In [3]:
print(sales.columns.tolist())

['tanggal', 'qty', 'nama_produk']


### Cek tanggal terlewat

In [4]:
# Pastikan kolom tanggal bertipe datetime
sales["tanggal"] = pd.to_datetime(sales["tanggal"], errors="coerce")

# Periode yang seharusnya
tanggal_mulai = pd.Timestamp("2022-01-01")
tanggal_akhir = pd.Timestamp("2024-12-31")

# Membuat seluruh tanggal yang seharusnya ada
semua_tanggal = pd.date_range(
    start=tanggal_mulai,
    end=tanggal_akhir,
    freq="D"
)

# Ambil tanggal yang benar-benar ada di sales
tanggal_sales = sales["tanggal"].dropna().dt.normalize().unique()

# Cari tanggal yang tidak ada
tanggal_terlewat = semua_tanggal[
    ~semua_tanggal.isin(tanggal_sales)
]

print("Jumlah tanggal yang seharusnya :", len(semua_tanggal))
print("Jumlah tanggal yang tersedia   :", len(tanggal_sales))
print("Jumlah tanggal terlewat        :", len(tanggal_terlewat))

Jumlah tanggal yang seharusnya : 1096
Jumlah tanggal yang tersedia   : 552
Jumlah tanggal terlewat        : 544


In [5]:
hasil_tanggal = pd.DataFrame({
    "tanggal_terlewat": tanggal_terlewat
})

display(hasil_tanggal)

,tanggal_terlewat
0,2022-01-02
1,2022-01-15
2,2022-02-08
3,2022-02-13
4,2022-02-18
...,...
539,2024-12-27
540,2024-12-28
541,2024-12-29
542,2024-12-30


### Cek produk duplikat pada tanggal yang sama

In [6]:
duplikat = sales[
    sales.duplicated(
        subset=["tanggal", "nama_produk"],
        keep=False
    )
].copy()

duplikat = duplikat.sort_values(
    ["tanggal", "nama_produk"]
)

print("Jumlah baris yang terlibat dalam duplikasi:",
      len(duplikat))

display(duplikat)

Jumlah baris yang terlibat dalam duplikasi: 770


,tanggal,qty,nama_produk
1,2022-01-01,7,media
8,2022-01-01,2,media
22,2022-01-03,16,brokoli_kuning
27,2022-01-03,24,brokoli_kuning
15,2022-01-03,1,media
...,...,...,...
3175,2024-11-22,15,media
3181,2024-11-24,5,media
3186,2024-11-24,1,media
3209,2024-12-04,1,media


In [7]:
rekap_duplikat = (
    sales
    .groupby(["tanggal", "nama_produk"])
    .size()
    .reset_index(name="jumlah")
)

rekap_duplikat = rekap_duplikat[
    rekap_duplikat["jumlah"] > 1
]

rekap_duplikat = rekap_duplikat.sort_values(
    ["tanggal", "nama_produk"]
)

print(
    "Jumlah kombinasi tanggal + produk yang duplikat:",
    len(rekap_duplikat)
)

display(rekap_duplikat)

Jumlah kombinasi tanggal + produk yang duplikat: 326


,tanggal,nama_produk,jumlah
4,2022-01-01,media,2
12,2022-01-03,brokoli_kuning,2
14,2022-01-03,media,4
16,2022-01-03,melati_mini,2
22,2022-01-04,media,3
...,...,...,...
2713,2024-11-17,cagak_pot,2
2718,2024-11-17,media,4
2734,2024-11-22,media,4
2740,2024-11-24,media,2


### Cek nama produk yang tidak ada di daftar produk

In [8]:
# Pastikan nama produk berupa string
sales["nama_produk"] = sales["nama_produk"].astype(str).str.strip()

# Bersihkan daftar produk TXT
daftar_produk = [
    produk.strip()
    for produk in daftar_produk
]

# Cari produk sales yang tidak ada dalam TXT
produk_tidak_dikenal = sorted(
    set(sales["nama_produk"]) - set(daftar_produk)
)

print(
    "Jumlah nama produk yang tidak ditemukan:",
    len(produk_tidak_dikenal)
)

for produk in produk_tidak_dikenal:
    print(produk)

Jumlah nama produk yang tidak ditemukan: 0


### Cleaning data type

In [9]:
# Pastikan tanggal menjadi datetime
sales["tanggal"] = pd.to_datetime(
    sales["tanggal"],
    errors="coerce"
)

# Bersihkan nama produk
sales["nama_produk"] = (
    sales["nama_produk"]
    .astype(str)
    .str.strip()
)

# Pastikan qty numerik
sales["qty"] = pd.to_numeric(
    sales["qty"],
    errors="coerce"
)

# Hapus baris dengan tanggal atau produk kosong
sales = sales.dropna(
    subset=["tanggal", "nama_produk"]
)

print(sales.shape)

(3211, 3)


### Menentukan peridoe aktual dari data

In [10]:
tanggal_mulai = pd.Timestamp("2022-01-01")
tanggal_akhir = sales["tanggal"].max()

print("Tanggal mulai :", tanggal_mulai)
print("Tanggal akhir :", tanggal_akhir)

Tanggal mulai : 2022-01-01 00:00:00
Tanggal akhir : 2024-12-04 00:00:00


### Gabungkan duplikasi data

In [11]:
sales_clean = (
    sales
    .groupby(
        ["tanggal", "nama_produk"],
        as_index=False
    )["qty"]
    .sum()
)

print("Jumlah data setelah menggabungkan duplikasi:",
      len(sales_clean))

Jumlah data setelah menggabungkan duplikasi: 2767


#### Cek lagi duplikasinya

In [12]:
cek_duplikat = sales_clean.duplicated(
    subset=["tanggal", "nama_produk"]
).sum()

print("Duplikasi tersisa:", cek_duplikat)

Duplikasi tersisa: 0


### Kombinasi tanggal x nama_produk

In [13]:
# Mengambil seluruh tanggal
semua_tanggal = pd.date_range(
    start=tanggal_mulai,
    end=tanggal_akhir,
    freq="D"
)

# Mengambil seluruh produk
semua_produk = sorted(
    sales_clean["nama_produk"].unique()
)

# Kombinasi nama produk untuk setiap tanggal yang terlewat
kombinasi = pd.MultiIndex.from_product(
    [semua_tanggal, semua_produk],
    names=["tanggal", "nama_produk"]
)

dataset = kombinasi.to_frame(index=False)

### Masukkan data penjualan asli

In [14]:
dataset = dataset.merge(
    sales_clean,
    on=["tanggal", "nama_produk"],
    how="left"
)

### Masukkan qty = 0

In [15]:
dataset["qty"] = dataset["qty"].fillna(0)

### Urutkan kembali dataset

In [16]:
dataset = dataset.sort_values(
    ["nama_produk", "tanggal"]
).reset_index(drop=True)

In [17]:
dataset

,tanggal,nama_produk,qty
0,2022-01-01,aglonema_rotundum_aceh,0.0
1,2022-01-02,aglonema_rotundum_aceh,0.0
2,2022-01-03,aglonema_rotundum_aceh,0.0
3,2022-01-04,aglonema_rotundum_aceh,0.0
4,2022-01-05,aglonema_rotundum_aceh,0.0
...,...,...,...
48100,2024-11-30,rumput_jepang,0.0
48101,2024-12-01,rumput_jepang,0.0
48102,2024-12-02,rumput_jepang,0.0
48103,2024-12-03,rumput_jepang,0.0


### Menambahkan product_id untuk memudahkan model

In [18]:
produk_unik = sorted(
    dataset["nama_produk"].unique()
)

product_to_id = {
    produk: i
    for i, produk in enumerate(produk_unik, start=1)
}

dataset["product_id"] = (
    dataset["nama_produk"]
    .map(product_to_id)
)

In [20]:
dataset

,tanggal,nama_produk,qty,product_id
0,2022-01-01,aglonema_rotundum_aceh,0.0,1
1,2022-01-02,aglonema_rotundum_aceh,0.0,1
2,2022-01-03,aglonema_rotundum_aceh,0.0,1
3,2022-01-04,aglonema_rotundum_aceh,0.0,1
4,2022-01-05,aglonema_rotundum_aceh,0.0,1
...,...,...,...,...
48100,2024-11-30,rumput_jepang,0.0,45
48101,2024-12-01,rumput_jepang,0.0,45
48102,2024-12-02,rumput_jepang,0.0,45
48103,2024-12-03,rumput_jepang,0.0,45


In [ ]:
dataset.to_csv('clean_dataset.csv', index=False)